# 02 · Clip raster against rasterio

**Question.** Does Fieldwork's `clip_raster` widget select the same cells as GDAL through `rasterio.mask`, for each of its inclusion methods?

**Fixture.** A synthetic 24 × 18 CRS84 grid with a deterministic value ramp, so a value disagreement is traceable to a cell rather than hidden in noise, and a concave cutline whose vertices sit **exactly on pixel corners**. Pixel-edge alignment is where rasterization conventions diverge, so the fixture aims at it deliberately.

**Independence.** Nothing here imports Fieldwork code. The clip is rebuilt from the GeoTIFF, the cutline and each case's options.

Run `00-kernel-check.ipynb` first. The design record and the write-up are in the Fieldwork repository: `docs/experiments/41-validation-lab.md` and `docs/experiments/29-raster-edge-inclusion.md`.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_02_clip import run

report = run(Path.cwd() / "fixtures" / "clip-all-touched-001.json")
print(json.dumps(report["environment"], indent=2))
for case in report["cases"]:
    o = case["options"]
    print(f"\n{o['method']} margin {o.get('marginPixels', 0)}")
    print(f"  reference {case['referenceIncluded']:4d} cells   fieldwork {case['fieldworkIncluded']:4d} cells")
    print(f"  included sets agree: {case['maskAgrees']}   window agrees: {case['windowAgrees']}")
    if case["marginModelledAs"]:
        print(f"  margin modelled as: {case['marginModelledAs']}")

## Where the two disagree

A count alone does not say whether a difference is systematic. The cell below asks two questions of every differing cell: which side included it, and whether it sits beside a cutline coordinate lying exactly on a pixel boundary.

In [ ]:
for case in report["cases"]:
    rows = case["differingCells"]
    if not rows:
        continue
    o = case["options"]
    reference_only = sum(1 for r in rows if r["reference"] and not r["fieldwork"])
    fieldwork_only = sum(1 for r in rows if r["fieldwork"] and not r["reference"])
    aligned = sum(1 for r in rows if r["besidePixelAlignedCutline"])
    print(f"{o['method']} margin {o.get('marginPixels', 0)}: {case['differingCellCount']} differing")
    print(f"  reference only {reference_only}, fieldwork only {fieldwork_only}")
    print(f"  beside a pixel-aligned cutline coordinate: {aligned} of {len(rows)} reported")

## Reading the result

**Cell centre inside agrees exactly** — same cropped window, same included cells, same retained values.

**All touched does not.** Fieldwork includes cells GDAL excludes and none the other way, and every difference sits beside a pixel-aligned cutline coordinate. The cause is Fieldwork's own definition: its inclusion test measures distance to a **closed** cell rectangle, so a cutline edge running exactly along the border between two cells touches both, while GDAL's `ALL_TOUCHED` assigns such an edge to one side. A 1e-9 pixel pad on the window widens the crop by a further cell.

The difference is one-sided, so an all-touched clip in Fieldwork retains more boundary cells than a GDAL-derived one. For a count-valued raster such as population that inflates any total computed from the clip, and a figure from one convention is not comparable with the other unless the convention is stated.

**The margin case is inconclusive by construction.** `rasterio` has no margin parameter, so the check models a one-pixel margin as a 3 × 3 binary dilation and labels it as a model. A margin defined as a Euclidean distance from a cell rectangle has no exact discrete analogue; the reported gap measures the distance to that model, not a validated disagreement.

## What this check does not establish

That either convention is the right one for a given analysis. GDAL is the reference implementation here, not an authority on what a public-health question needs. It covers one synthetic grid, one cutline, CRS84 only, one band and float64 values; nothing about reprojection, resampling, multi-band rasters, compression or large-file behaviour is tested.